In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
import warnings
warnings.filterwarnings("ignore")

In [16]:
df = pd.read_csv('mumbai-house-price-data-cleaned.csv')

In [17]:
df.head()

,title,price,area,price_per_sqft,locality,city,property_type,bedroom_num,bathroom_num,balcony_num,furnished,age,total_floors,latitude,longitude
0,Octave Parijas Horizon,6600283,757,8719.000000,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,0,1,19.244410,73.123253
1,Shakti Siyara Heights,6169841,652,9462.946319,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,0,1,19.257294,73.148872
2,Bhagwati Bhagwati Celeste,4599936,396,11616.000000,Dombivali,Mumbai,Apartment,1,1,0,Unfurnished,0,1,19.209026,73.081276
3,Relcon Ridhi Sidhi Sadan Of Ridhi Sidhi Co Ope...,51980000,1130,46000.000000,Ville Parle,Mumbai,Apartment,3,3,0,Unfurnished,0,1,19.097841,72.851158
4,J P Ruchita Bliss,3915000,435,9000.000000,Nala Sopara,Mumbai,Apartment,1,1,0,Unfurnished,0,1,19.420601,72.809319


In [18]:
p_low  = df["price"].quantile(0.01)
p_high = df["price"].quantile(0.99)
df = df[(df["price"] >= p_low) & (df["price"] <= p_high)]

In [19]:
df = df[(df["area"] >= 150) & (df["area"] <= 10_000)]
 
# Keep only BHK 1-8 (removes studio=0 and unrealistic 15-BHK)
df = df[(df["bedroom_num"] >= 1) & (df["bedroom_num"] <= 8)]

In [20]:
FEATURES = ["area", "locality", "property_type", "bedroom_num"]
TARGET   = "price"
df = df.dropna(subset=FEATURES + [TARGET])
 
print(f"[2/6] After cleaning  →  {df.shape[0]:,} rows remain")

[2/6] After cleaning  →  70,491 rows remain


In [21]:
X = df[FEATURES].copy()
y = df[TARGET].copy()
 
# Log-transform target to reduce skew (helps RF too)
y_log = np.log1p(y)
 
# ── 4. Preprocessing ──────────────────────────────────────────────────────────
#   - Numeric:      area, bedroom_num  →  StandardScaler
#   - Categorical:  locality, property_type  →  OrdinalEncoder
 
numeric_features     = ["area", "bedroom_num"]
categorical_features = ["locality", "property_type"]
 
# Collect all unique locality values for the dashboard
localities = sorted(X["locality"].unique().tolist())
property_types = sorted(X["property_type"].unique().tolist())
print(f"[3/6] Unique localities: {len(localities)}  |  Property types: {property_types}")
 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                ),
         categorical_features),
    ]
)
 

[3/6] Unique localities: 388  |  Property types: ['Apartment', 'Independent Floor', 'Independent House', 'Villa']


In [22]:
rf_model = RandomForestRegressor(
    n_estimators=200,        # 200 trees — good accuracy / speed balance
    max_depth=20,            # Prevents over-fitting on noisy locality data
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",     # Classic RF setting
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)
 
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model",        rf_model)
])

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
 
print(f"\n[4/6] Training Random Forest on {len(X_train):,} samples …")
pipeline.fit(X_train, y_train)
print("      ✓ Training complete!")
 
# Evaluate on test set (convert back from log space)
y_pred_log = pipeline.predict(X_test)
y_pred      = np.expm1(y_pred_log)
y_true      = np.expm1(y_test)
 
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
 
print(f"\n[5/6] Test Set Performance:")
print(f"      MAE   = ₹{mae:>15,.0f}")
print(f"      RMSE  = ₹{rmse:>15,.0f}")
print(f"      R²    =  {r2:.4f}  ({r2*100:.1f}% variance explained)")


[4/6] Training Random Forest on 56,392 samples …
      ✓ Training complete!

[5/6] Test Set Performance:
      MAE   = ₹      3,837,145
      RMSE  = ₹      7,609,029
      R²    =  0.8145  (81.4% variance explained)


In [24]:
feature_names = numeric_features + categorical_features
importances   = pipeline.named_steps["model"].feature_importances_
print("\n      Feature Importances:")
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"      {name:<18} {imp:.4f}  {bar}")


      Feature Importances:
      bedroom_num        0.4000  ████████████████
      locality           0.2990  ███████████
      area               0.2901  ███████████
      property_type      0.0109  


In [25]:
with open("model.pkl", "wb") as f:
    pickle.dump(pipeline, f)
print("\n[6/6] Saved  →  model.pkl")
 
meta = {
    "localities":     localities,
    "property_types": property_types,
    "bhk_min": int(X["bedroom_num"].min()),
    "bhk_max": int(X["bedroom_num"].max()),
    "area_min": int(X["area"].min()),
    "area_max": int(X["area"].max()),
    "area_median": int(X["area"].median()),
}
with open("meta.pkl", "wb") as f:
    pickle.dump(meta, f)
print("       Saved  →  meta.pkl\n")
print("=" * 60)
print("  Pipeline ready. Run the dashboard with:")
print("  streamlit run dashboard.py")
print("=" * 60)


[6/6] Saved  →  model.pkl
       Saved  →  meta.pkl

  Pipeline ready. Run the dashboard with:
  streamlit run dashboard.py
